# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [8]:
%pip install -qU langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install -qU jq ijson langchain langchain-classic langchain-elasticsearch langchain-openai tiktoken

Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY  = os.getenv("ELASTIC_API_KEY")
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")

# sanity checks (won't print your keys, just confirms they're present)
assert ELASTIC_CLOUD_ID, "Missing ELASTIC_CLOUD_ID in .env"
assert ELASTIC_API_KEY,  "Missing ELASTIC_API_KEY in .env"
assert OPENAI_API_KEY,   "Missing OPENAI_API_KEY in .env"

In [3]:
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="chatbot_rag_index_v2",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [15]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name", "")
    metadata["summary"] = record.get("summary", "")
    metadata["url"] = record.get("url", "")
    metadata["category"] = record.get("category", "")
    metadata["updated_at"] = record.get("updated_at", "")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=100 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

In [16]:
from datetime import datetime

def normalize_updated_at(docs):
    for d in docs:
        v = d.metadata.get("updated_at", None)

        # Remove empty/None updated_at so ES doesn't try to parse it as a date
        if v is None or (isinstance(v, str) and v.strip() == ""):
            d.metadata.pop("updated_at", None)
            continue

        # If it's already a datetime, keep it
        if isinstance(v, datetime):
            continue

        # If it's a string, try to parse ISO-ish; if parsing fails, remove it
        if isinstance(v, str):
            try:
                # handles "2024-01-15", "2024-01-15T12:34:56", etc.
                d.metadata["updated_at"] = datetime.fromisoformat(v.replace("Z", "+00:00"))
            except Exception:
                d.metadata.pop("updated_at", None)

normalize_updated_at(docs)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [20]:
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI


embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore.from_documents(
    docs,
    embedding=embeddings,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="chatbot_rag_index_v2",
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [21]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
import logging

def format_document(doc, prompt: PromptTemplate) -> str:
    # doc.page_content + doc.metadata are what we need
    return prompt.format(page_content=doc.page_content, **doc.metadata)

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """
You are an assistant for question-answering tasks.
Use ONLY the following retrieved context to answer the question.
If you don't know the answer, just say you don't know.

When answering:
- Be clear and structured.
- Cite sources using the SOURCE name provided.
- If multiple sources are used, list them at the end.

Context:
{context}

Question:
{question}

Answer:
"""
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

---- Answer ----
content='The NASA sales team refers to the North America South America region within the sales organization. This region includes the United States, Canada, Mexico, as well as Central and South America. The NASA region is led by two Area Vice-Presidents: Laura Martinez, who is the Area Vice-President of North America, and Gary Johnson, who is the Area Vice-President of South America. The sales team in this region is responsible for identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction across these areas. \n\n(SOURCE: Sales Organization Overview)' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 1853, 'total_tokens': 1966, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_

In [22]:
ans = chain.invoke("How is the sales organization structured?")
print(ans.content)

The sales organization is structured into four main regions, each led by an Area Vice-President. The regions are as follows:

1. **The Americas**: This region includes the United States, Canada, Mexico, and Central and South America. It is managed by two Area Vice-Presidents:
   - Laura Martinez (North America)
   - Gary Johnson (South America)

2. **Europe**: The European sales team covers the entire continent, including countries like the United Kingdom, Germany, France, Spain, and Italy. The Area Vice-President for Europe is Rajesh Patel.

3. **Asia-Pacific**: This region includes countries such as China, Japan, South Korea, India, Australia, and New Zealand. Mei Li serves as the Area Vice-President for Asia-Pacific.

4. **Middle East & Africa**: This region comprises countries across the Middle East and Africa, including the United Arab Emirates, Saudi Arabia, South Africa, and Nigeria. Jamal Abdi is the Area Vice-President for this region.

Each regional team consists of dedicated

**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?